# 01 数据清洗：Online Retail II

**输入**：`data/raw/online_retail_II.parquet`（1,067,371 行，原始数据只读不改）

**输出**：
- `data/clean/transactions.parquet` —— 正常交易表（后续 RFM / 流失建模的主表）
- `data/clean/returns.parquet` —— 退货表（用于退货率分析）

**原则**：每条规则先看证据（这批数据长什么样）再做决策（为什么这么处理），对应 `docs/cleaning_rules.md` 的规则 #1–#6。

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent  # 从 notebooks/ 目录运行时退回项目根
RAW, CLEAN = ROOT / "data" / "raw", ROOT / "data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(RAW / "online_retail_II.parquet")
print(f"原始数据 {len(df):,} 行")
df.head()

原始数据 1,067,371 行


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 规则 1：退货单拆分（Invoice 以 C 开头）

退货不是噪音，是业务信号：单独成表用于退货率分析，不与正常交易混算 GMV。

In [2]:
is_return = df["Invoice"].str.startswith("C", na=False)
returns = df[is_return].copy()
sales = df[~is_return].copy()
print(f"退货单 {len(returns):,} 行（占全表 {len(returns) / len(df):.1%}），正常交易 {len(sales):,} 行")
returns["InvoiceDate"].dt.year.value_counts().sort_index()

退货单 19,494 行（占全表 1.8%），正常交易 1,047,877 行


InvoiceDate
2009    1015
2010    9919
2011    8560
Name: count, dtype: int64

## 规则 2 & 3：非退货单中的 Quantity ≤ 0 / Price ≤ 0

退货单已被拆走，剩下的负数/零值属于什么？看构成再决定。

In [3]:
neg_q = sales["Quantity"] <= 0
neg_p = sales["Price"] <= 0
print(f"非退货单中 Quantity≤0：{neg_q.sum():,} 行；Price≤0：{neg_p.sum():,} 行")
sales[neg_q][["Invoice", "StockCode", "Description", "Quantity", "Price"]].head(8)

非退货单中 Quantity≤0：3,457 行；Price≤0：6,207 行


,Invoice,StockCode,Description,Quantity,Price
263,489464,21733,85123a mixed,-96,0.0
283,489463,71477,short,-240,0.0
284,489467,85123A,21733 mixed,-192,0.0
470,489521,21646,<NA>,-50,0.0
3114,489655,20683,<NA>,-44,0.0
3162,489660,35956,lost,-1043,0.0
3168,489663,35605A,damages,-117,0.0
4296,489806,18010,<NA>,-770,0.0


In [4]:
print("Price≤0 行的 StockCode 构成（前 10）：")
sales[neg_p]["StockCode"].value_counts().head(10)

Price≤0 行的 StockCode 构成（前 10）：


StockCode
46000M    18
22501     18
79321     17
21116     16
22423     16
23084     16
46000S    15
22734     15
22139     14
35965     14
Name: count, dtype: int64[pyarrow]

## 规则 4：Customer_ID 缺失（全表 22.8%）

**决策**：交易表**保留**这些行（它们是真实销售额，删掉会低估 GMV），打 `has_customer` 标记；客户维度分析（RFM、流失）时再剔除。缺失多半是未登录/游客下单，这本身就是一个业务洞察的线索。

In [5]:
sales["Amount"] = sales["Quantity"] * sales["Price"]
no_id = sales["Customer_ID"].isna()
gmv = sales["Amount"].sum()
print(f"正常交易中缺失 Customer_ID：{no_id.sum():,} 行（{no_id.mean():.1%}）")
print(f"这些行贡献销售额：{sales.loc[no_id, 'Amount'].sum() / gmv:.1%}")

正常交易中缺失 Customer_ID：242,257 行（23.1%）
这些行贡献销售额：14.8%


## 规则 5：完全重复行

In [6]:
dup = sales.duplicated()
print(f"完全重复行：{dup.sum():,} 行（{dup.mean():.2%}）")

完全重复行：33,945 行（3.24%）


## 规则 6：异常值与非商品交易

看长尾分布、无数字的 StockCode（POST 邮费 / M 手工调账 / S 样品等非商品交易）、以及国家分布。**长尾不删**，只标记。

In [7]:
print(sales[["Quantity", "Price", "Amount"]].describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]).round(2).to_string())
non_product = ~sales["StockCode"].str.contains(r"\d", na=False)
print(f"\n非商品交易（StockCode 不含数字）：{non_product.sum():,} 行，占销售额 {sales.loc[non_product, 'Amount'].sum() / gmv:.2%}")
sales.loc[non_product, "StockCode"].value_counts().head(10)

         Quantity       Price      Amount
count  1047877.00  1047877.00  1047877.00
mean        10.59        3.90       19.86
std        135.28       94.36      217.47
min      -9600.00   -53594.36   -53594.36
0.1%       -61.00        0.00        0.00
1%           1.00        0.19        0.42
50%          3.00        2.10        9.95
99%        108.00       18.00      183.60
99.9%      500.00      157.93      836.14
max      80995.00    25111.09   168469.60



非商品交易（StockCode 不含数字）：4,382 行，占销售额 3.30%


StockCode
POST            1893
DOT             1443
M                883
ADJUST            36
BANK CHARGES      35
DCGSSGIRL         25
DCGSSBOY          23
PADS              18
B                  6
m                  5
Name: count, dtype: int64[pyarrow]

In [8]:
by_country = sales.groupby("Country")["Amount"].sum().sort_values(ascending=False)
print(f"UK 占销售额：{by_country.get('United Kingdom', 0) / by_country.sum():.1%}")
print(by_country.head(8).round(0).to_string())
print(f"\nUnspecified 国家：{(sales['Country'] == 'Unspecified').sum():,} 行（保留，不单独处理）")

UK 占销售额：85.1%
Country
United Kingdom    17712302.0
EIRE                664432.0
Netherlands         554232.0
Germany             431262.0
France              356945.0
Australia           169968.0
Spain               109179.0
Switzerland         101011.0

Unspecified 国家：752 行（保留，不单独处理）


## 生成清洗后数据

| 决策 | 处理 |
|---|---|
| 退货单 | 单独成表 returns.parquet |
| 非退货单 Quantity≤0 / Price≤0 | 剔除（无业务含义的负值/零值，数量已在上方统计） |
| 完全重复行 | 剔除 |
| Customer_ID 缺失 | 保留 + `has_customer` 标记 |
| 非商品交易 | 保留 + `is_product` 标记 |
| 长尾极值 | 保留（分析时用分位数/对数轴） |

In [9]:
transactions = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0) & ~dup].copy()
transactions["has_customer"] = transactions["Customer_ID"].notna()
transactions["is_product"] = transactions["StockCode"].str.contains(r"\d", na=False)

transactions.to_parquet(CLEAN / "transactions.parquet", index=False)
returns.to_parquet(CLEAN / "returns.parquet", index=False)
print(f"transactions：{len(transactions):,} 行（占原始 {len(transactions) / len(df):.1%}）")
print(f"returns：{len(returns):,} 行")

transactions：1,007,913 行（占原始 94.4%）
returns：19,494 行


In [10]:
summary = {
    "规则1 退货单拆出": len(returns),
    "规则2 非退货单 Quantity≤0 剔除": int(neg_q.sum()),
    "规则3 非退货单 Price≤0 剔除": int(neg_p.sum()),
    "规则4 Customer_ID 缺失（保留+标记）": int(no_id.sum()),
    "规则5 完全重复行剔除": int(dup.sum()),
    "规则6 非商品交易（保留+标记）": int(non_product.sum()),
}
for k, v in summary.items():
    print(f"{k}: {v:,} 行")
print(f"\n最终交易表 {len(transactions):,} 行 / 原始 {len(df):,} 行")

规则1 退货单拆出: 19,494 行
规则2 非退货单 Quantity≤0 剔除: 3,457 行
规则3 非退货单 Price≤0 剔除: 6,207 行
规则4 Customer_ID 缺失（保留+标记）: 242,257 行
规则5 完全重复行剔除: 33,945 行
规则6 非商品交易（保留+标记）: 4,382 行

最终交易表 1,007,913 行 / 原始 1,067,371 行
